# 04 - Memory Management & Troubleshooting

This notebook covers Spark's memory internals and common failure patterns:

1. **Memory model** - Unified Memory Manager, execution vs storage memory, dynamic borrowing
2. **OOM patterns** - driver OOM (collect/toPandas), executor OOM (skew, large broadcast)
3. **Tungsten & codegen** - binary format, whole-stage codegen, why Python UDFs hurt codegen and what alternatives exist

Dataset: NYC Yellow Taxi Trip Records (2024).

In [2]:
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
import os, time


def _root():
    env = os.getenv("SPARK_TUNING_PROJECT_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise RuntimeError("Set SPARK_TUNING_PROJECT_ROOT or run from the repo root")


PROJECT_ROOT = _root()
DATA_DIR = Path(os.getenv("SPARK_TUNING_DATA_DIR", str(PROJECT_ROOT / "data"))).resolve()
TAXI_DATA_DIR = DATA_DIR / "taxi"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

PROJECT_ROOT: /
DATA_DIR: /data


In [3]:
taxi = spark.read.parquet(str(TAXI_DATA_DIR))
print("rows:", taxi.count())
print("partitions:", taxi.rdd.getNumPartitions())

rows: 41169720
partitions: 12


Spark UI: __http://localhost:4040__

We use Spark UI in this lab after each action to validate:
* execution vs storage memory usage per executor
* spill (memory/disk) in task metrics
* GC time pressure
* whether driver or executor failed (OOM pattern)

In [4]:
# Keep plans deterministic for learning cells
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("spark.sql.adaptive.enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

spark.sql.adaptive.enabled: false
spark.sql.shuffle.partitions: 8


## Helpers

Reusable functions for tracking jobs/stages in Spark UI and inspecting physical plans.

In [5]:
def run_and_report(action_label, action_fn):
    """Executes an action under a unique job group ID (visible in Spark UI), then collects and prints the job and stage IDs it produced."""
    sc = spark.sparkContext
    tracker = sc.statusTracker()
    group_id = f"notebook_demo_{int(time.time() * 1000)}"
    sc.setJobGroup(group_id, action_label)
    t0 = time.perf_counter()
    try:
        result = action_fn()
    finally:
        dt = time.perf_counter() - t0
        job_ids = list(tracker.getJobIdsForGroup(group_id))
        stage_ids = set()
        for job_id in job_ids:
            job_info = tracker.getJobInfo(job_id)
            if job_info is not None:
                stage_ids.update(list(job_info.stageIds))
        print(
            f"{action_label} -> {dt:.2f}s jobs={job_ids}, stages={sorted(stage_ids)}, stage_count={len(stage_ids)}"
        )
        sc.setJobGroup("", "")
    return result, dt


def show_nodes(df, keys=("Exchange", "BroadcastHashJoin", "SortMergeJoin", "BatchEvalPython", "AdaptiveSparkPlan")):
    plan = df._jdf.queryExecution().executedPlan().toString()
    for line in plan.splitlines():
        if any(k in line for k in keys):
            print(line.strip())


def show_exchange_nodes(df):
    plan = df._jdf.queryExecution().executedPlan().toString()
    lines = [line.strip() for line in plan.splitlines() if "Exchange" in line]
    if not lines:
        print("No Exchange nodes in executed physical plan.")
    else:
        print("Exchange-related nodes:")
        for line in lines:
            print(line)

## Section 1 - Spark Memory Model and UMM

### Unified Memory Manager (UMM)

Each executor's JVM heap is divided into:
- **Reserved memory** (300 MB) - fixed, internal Spark overhead
- **User memory** (1 - `spark.memory.fraction`) - user data structures, UDF variables, RDD metadata
- **Unified memory** (`spark.memory.fraction`, default 0.6) - shared pool split into execution and storage memory

**Execution memory** - buffers for sort, shuffle, join hash tables, aggregation. When operators exceed available space, Spark spills to disk from here.

**Storage memory** - cache/persist blocks and broadcast variables.

Key concepts:
- **JVM heap** - memory allocated by Java at executor startup, sized by `spark.executor.memory`. Everything Spark does in an executor lives here
- **GC (Garbage Collector)** - Java's automatic memory cleanup. While GC runs, the executor pauses (**stop-the-world**). More cached objects on heap => more GC pressure => slower tasks
- **memoryOverhead** - memory outside JVM heap for Python workers, network buffers, container overhead. Not subject to GC.
- **Off-heap** - additional memory outside JVM, disabled by default (`spark.memory.offHeap.enabled`). Tungsten can store data here, bypassing GC entirely

### Dynamic borrowing and eviction

- execution can borrow from idle storage
- storage can use idle execution space
- but execution can **evict** storage blocks when it needs memory back - storage cannot evict execution
- this means **execution always has priority** => if a sort or shuffle needs memory, cached data gets dropped first
- **protected zone**: the bottom `spark.memory.storageFraction` (default 0.5) of unified memory is never evicted - execution can only reclaim storage **above** this line. If all cached data fits within the protected zone, eviction never happens regardless of execution pressure. On a 2 GB executor: unified = ~1049 MB, protected storage = ~524 MB
- evicted block fate depends on `StorageLevel`: `MEMORY_ONLY` drops them entirely - next read recomputes from source (slower AFTER). `MEMORY_AND_DISK` (the default `.cache()`) spills them to disk instead - next read hits disk, not memory, but no recompute

### When things go wrong

- **High GC Time in Spark UI** - too much cached data on heap, or many short-lived objects from wide operations (e.g. `groupBy` on millions of keys creates temporary hash maps that GC must clean up). **Fix**: increase heap, reduce cache, or use serialized storage level.
- **Spill (memory -> disk)** - execution memory full, but Spark handles it gracefully by writing to disk. Job finishes, just slower. **Fix**: reduce partition skew or increase partitions.
- **OOM** - heap completely exhausted, spill cannot save it (e.g. single object too large to fit, or memory fragmented beyond recovery). Executor or driver crashes. **Fix**: distinguish driver vs executor OOM (next section).

### Demo - storage memory under execution pressure

Cache a DataFrame, then run a heavy aggregation + sort that forces execution memory to spill.

**What to watch in Spark UI while pressure runs:**
- **Storage tab** -> all 24 cached partitions stay green - the cache survived
- **Jobs -> any Stage -> Spill columns** -> non-zero values confirm execution was genuinely stressed, but it spilled to disk rather than evicting the cache

This is correct behavior: a well-sized cache fits within the protected zone (~524 MB on this cluster) and is never evicted regardless of execution pressure. Execution spills instead - job is slower, cache is intact.

In [5]:
spark.catalog.clearCache()

cached_mid = (
    taxi.limit(5_000_000)
    .select(
        "PULocationID", "DOLocationID", "fare_amount", "tip_amount",
        "total_amount", "trip_distance", "passenger_count",
        "tpep_pickup_datetime", "tpep_dropoff_datetime"
    )
    .repartition(24, "PULocationID")
    .cache()
)

run_and_report("cache materialize", lambda: cached_mid.count())

[Stage 6:======================================================>  (23 + 1) / 24]

cache materialize -> 5.35s jobs=[3], stages=[4, 5, 6, 7], stage_count=4


(5000000, 5.3489440440000635)

In [6]:
run_and_report("cache read BEFORE pressure", lambda: cached_mid.count())

cache read BEFORE pressure -> 0.05s jobs=[4], stages=[8, 9, 10, 11], stage_count=4


(5000000, 0.05233704200009015)

In [10]:
# Force all 40M taxi rows into a single sort task by setting shuffle.partitions=1.
# tail(1) requires the full sort to execute - cannot be optimized away.
# This reliably triggers spill (same pattern as the spill demo in notebook 05).
# While running: Spark UI -> Storage tab should show all cached partitions still present.
_partitions_bkp = spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.shuffle.partitions", "1")

pressure = taxi.orderBy("total_amount", "tip_amount", "trip_distance", "fare_amount")

run_and_report("execution pressure", lambda: pressure.tail(1))

spark.conf.set("spark.sql.shuffle.partitions", _partitions_bkp)

[Stage 25:>                                                         (0 + 1) / 1]

execution pressure -> 44.17s jobs=[8], stages=[24, 25], stage_count=2


In [11]:
run_and_report("cache read AFTER pressure", lambda: cached_mid.count())

cache read AFTER pressure -> 0.06s jobs=[9], stages=[26, 27, 28, 29], stage_count=4


(5000000, 0.06393883400005507)

## Section 2 - OOM Patterns: Driver vs Executor

OOM crashes either the driver or an executor. Location determines the fix.

**Driver OOM** - driver is a single JVM process, heap set by `spark.driver.memory` (default 1 GB). Triggered by:
- `collect()` or `toPandas()` pulling all rows to the driver process
- broadcasting a table that doesn't fit in executor memory (Driver collects broadcast before sending to executors)

Fix: aggregate in the cluster first, collect only the small result. Use `take(n)` or `limit(n).collect()` for sampling.

**Executor OOM** - each executor JVM has `spark.executor.memory`. Triggered by:
- **Partition skew** - one task processes 10-100× more data than others, overflows its execution memory slice
- **Under-partitioned shuffle** - too few post-shuffle partitions, each partition holds too many rows
- **Oversized broadcast** - build side replicated to every executor simultaneously
- **Under-partitioned input** - `coalesce(2)` on 41M rows gives 2 tasks each handling ~20M rows

Fix: check key distribution and task input sizes before touching memory configs. Spark UI -> Stages -> Tasks -> sort by Input Size to find the outlier.

**Spill is not OOM**: spill = execution memory full, Spark writes temp data to disk, job finishes (slower). OOM = heap exhausted, executor crashes. Spill appears in Spark UI task metrics as "Spill (Memory)" and "Spill (Disk)".

**Diagnostics order**:
1. Spark UI -> Executors: which executor died?
2. Stages -> Tasks: find the outlier by input size or shuffle read size
3. Fix data shape first (partitioning, skew), then tune memory knobs

### Demo A - driver: safe access vs. full collect

Estimate how much data a full `collect()` would pull to the driver, then show the safe alternatives. Check Spark UI -> Jobs: `take(20)` triggers one short job that reads only the first partition.

In [12]:
risk = taxi.select("PULocationID", "DOLocationID", "fare_amount", "tip_amount", "total_amount")
row_count = risk.count()
est_mb = row_count * 5 * 8 / 1024 / 1024
print(f"rows: {row_count:,}  estimated collect payload: ~{est_mb:.0f} MB")
print(f"spark.driver.memory: {spark.conf.get('spark.driver.memory', '1g')}  - a full collect on this frame risks OOM")

rows: 41,169,720  estimated collect payload: ~1571 MB
spark.driver.memory: 1g  - a full collect on this frame risks OOM


In [13]:
rows, _ = run_and_report("safe take(20)", lambda: risk.take(20))
print(f"rows returned: {len(rows)}")

agg_result, _ = run_and_report(
    "aggregate then collect",
    lambda: risk.groupBy("PULocationID")
               .agg(F.avg("total_amount").alias("avg_total"), F.count("*").alias("trips"))
               .orderBy(F.desc("trips"))
               .limit(10)
               .collect()
)
for row in agg_result:
    print(f"  zone {row.PULocationID:3d}  trips={row.trips:,}  avg={row.avg_total:.2f}")

safe take(20) -> 0.09s jobs=[11], stages=[32], stage_count=1
rows returned: 20
aggregate then collect -> 0.54s jobs=[12], stages=[33, 34], stage_count=2
  zone 132  trips=1,989,994  avg=76.53
  zone 237  trips=1,915,441  avg=20.46
  zone 161  trips=1,914,607  avg=24.57
  zone 236  trips=1,729,896  avg=20.76
  zone 162  trips=1,418,847  avg=23.99
  zone 230  trips=1,391,159  avg=27.99
  zone 186  trips=1,362,156  avg=24.83
  zone 142  trips=1,314,690  avg=21.87
  zone 138  trips=1,295,508  avg=66.45
  zone 170  trips=1,176,583  avg=23.75


### Demo B - executor: partition skew

Partition skew as an OOM trigger is covered in depth in notebook 03 (shuffle, joins, partitioning) - including salt-based mitigation and Spark UI task metrics. Refer to that notebook for the full demo.

### Demo C - executor: broadcast OOM risk

Join a large fact table with a small deduplicated location table two ways:

1. **BHJ with broadcast hint** - the small side (263 rows) is replicated to every executor via `BroadcastExchange`. No shuffle on either side. Fast and memory-safe because the broadcast payload is tiny.
2. **SMJ fallback** - broadcast disabled via `autoBroadcastJoinThreshold = -1`. Spark falls back to SortMergeJoin, which shuffles and sorts both sides. Each executor must buffer its share of the large fact table in execution memory during the sort phase.

The SMJ on this cluster triggers executor OOM - the sort buffers exhaust the 2 GB heap. This is the broadcast OOM risk in reverse: **not broadcasting when you should** forces a heavier join strategy that can kill executors on memory-constrained clusters.

**The second `run_and_report` cell will crash the executor and fail.** After it fails, restart the kernel (Kernel -> Restart Kernel) and re-run the setup cells before continuing.

In [6]:
fact = taxi.select("PULocationID", "DOLocationID", "total_amount")
small = taxi.select("PULocationID", "RatecodeID", "Airport_fee").dropDuplicates(["PULocationID"])
print(f"broadcast side rows: {small.count()} - safe to broadcast\n")

bhj = fact.join(F.broadcast(small), on="PULocationID", how="inner")
print("BHJ plan nodes:")
show_nodes(bhj)

broadcast side rows: 263 - safe to broadcast

BHJ plan nodes:
+- *(3) BroadcastHashJoin [PULocationID#7], [PULocationID#61], Inner, BuildRight, false
+- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, true] as bigint)),false), [plan_id=163]
+- Exchange hashpartitioning(PULocationID#61, 8), ENSURE_REQUIREMENTS, [plan_id=159]


In [ ]:
_, t_bhj = run_and_report("BHJ (broadcast hint)", lambda: bhj.count())

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
smj = fact.join(small, on="PULocationID", how="inner")
print("\nSMJ plan nodes (broadcast disabled):")
show_nodes(smj)
_, t_smj = run_and_report("SMJ fallback", lambda: smj.count())
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
print(f"\nBHJ: {t_bhj:.2f}s  SMJ: {t_smj:.2f}s")

## Section 3 - Tungsten, Whole-Stage Codegen, and UDF Cost

**Tungsten** replaces the generic Java object model with compact binary in-memory representation. A boxed Java `Long` costs 16-24 bytes; Tungsten stores it as 8 raw bytes. No GC pressure on that memory. Off-heap mode (`spark.memory.offHeap.enabled`) takes this further - data lives outside the JVM heap entirely.

**Whole-stage codegen (WSCG)** compiles operator pipelines (Filter + Project + Aggregate) into a single JVM method at runtime. In the physical plan, WSCG operators have a `*` prefix: `*(1) HashAggregate`. Data flows as CPU register values between operators - no intermediate row objects, no virtual dispatch overhead.

**Python UDF boundary** - a Python UDF runs in the Python interpreter, not in the JVM. Spark must:
1. Serialize each row JVM -> Python process (pickle, row by row)
2. Execute the Python function
3. Serialize the result back

This creates `BatchEvalPython` in the plan. WSCG cannot fuse across this node. Typical overhead: **5-10× slower** than an equivalent built-in SQL function.

**Pandas UDF** uses Apache Arrow for columnar batch transfer - one partition per round trip instead of one row. Still breaks WSCG (appears as `ArrowEvalPython`) but 10-50× faster than a Python UDF for the same operation.

Rule: built-in `pyspark.sql.functions` -> `@pandas_udf` -> `@udf` (last resort, only on small/filtered frames).

### Demo - built-in vs Python UDF vs Pandas UDF

Compute `tip/fare` ratio three ways. Check the physical plan for `BatchEvalPython` (row-by-row boundary) vs `ArrowEvalPython` (columnar batch) vs clean built-in execution.

In [12]:
print("Codegen and Tungsten settings:")
for k in ["spark.sql.codegen.wholeStage", "spark.memory.offHeap.enabled",
          "spark.sql.execution.arrow.pyspark.enabled"]:
    try:
        print(f"  {k} = {spark.conf.get(k)}")
    except Exception:
        print(f"  {k} = <unset>")
print()

base = taxi.select("PULocationID", "fare_amount", "tip_amount").filter(F.col("fare_amount") > 0)

builtin_df = base.withColumn("r", F.col("tip_amount") / F.col("fare_amount"))

@udf("double")
def tip_ratio_py(tip, fare):
    if fare is None or fare == 0 or tip is None: return 0.0
    return float(tip) / float(fare)

pyudf_df = base.withColumn("r", tip_ratio_py(F.col("tip_amount"), F.col("fare_amount")))

print("Built-in plan nodes (no BatchEvalPython):")
show_nodes(builtin_df)
print("\nPython UDF plan nodes (BatchEvalPython breaks WSCG):")
show_nodes(pyudf_df)

Codegen and Tungsten settings:
  spark.sql.codegen.wholeStage = true
  spark.memory.offHeap.enabled = <unset>
  spark.sql.execution.arrow.pyspark.enabled = true

Built-in plan nodes (no BatchEvalPython):

Python UDF plan nodes (BatchEvalPython breaks WSCG):
+- BatchEvalPython [tip_ratio_py(tip_amount#13, fare_amount#10)#118], [pythonUDF0#120]


In [13]:
# Cache base to isolate UDF cost from parquet I/O.
# agg(sum("r")) forces Spark to evaluate "r" on every row - count() does not.
spark.catalog.clearCache()
base.cache()
base.count()  # materialize

_, t_b = run_and_report("builtin",    lambda: builtin_df.agg(F.sum("r")).collect())
_, t_p = run_and_report("python udf", lambda: pyudf_df.agg(F.sum("r")).collect())
print(f"\nbuiltin: {t_b:.2f}s  python_udf: {t_p:.2f}s  factor: {t_p/max(t_b,1e-9):.1f}x slower")

26/03/25 22:22:29 WARN MemoryStore: Not enough space to cache rdd_70_1 in memory! (computed 45.7 MiB so far)
26/03/25 22:22:29 WARN MemoryStore: Not enough space to cache rdd_70_6 in memory! (computed 45.8 MiB so far)
26/03/25 22:22:29 WARN MemoryStore: Not enough space to cache rdd_70_4 in memory! (computed 45.7 MiB so far)
26/03/25 22:22:29 WARN MemoryStore: Not enough space to cache rdd_70_3 in memory! (computed 45.8 MiB so far)
26/03/25 22:22:29 WARN BlockManager: Persisting block rdd_70_3 to disk instead.
26/03/25 22:22:29 WARN BlockManager: Persisting block rdd_70_1 to disk instead.
26/03/25 22:22:29 WARN BlockManager: Persisting block rdd_70_4 to disk instead.
26/03/25 22:22:29 WARN BlockManager: Persisting block rdd_70_6 to disk instead.
26/03/25 22:22:29 WARN MemoryStore: Not enough space to cache rdd_70_2 in memory! (computed 45.8 MiB so far)
26/03/25 22:22:29 WARN BlockManager: Persisting block rdd_70_2 to disk instead.
26/03/25 22:22:30 WARN MemoryStore: Not enough space to

builtin -> 0.27s jobs=[10], stages=[21, 22], stage_count=2


[Stage 23:======================================>                  (8 + 4) / 12]

python udf -> 5.13s jobs=[11], stages=[23, 24], stage_count=2

builtin: 0.27s  python_udf: 5.13s  factor: 18.9x slower


In [14]:
try:
    import pandas as pd
    from pyspark.sql.functions import pandas_udf

    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

    @pandas_udf("double")
    def tip_ratio_pd(t: pd.Series, f: pd.Series) -> pd.Series:
        ff = f.replace(0, pd.NA)
        return (t.astype("float64") / ff.astype("float64")).fillna(0.0)

    pdu_df = base.withColumn("r", tip_ratio_pd(F.col("tip_amount"), F.col("fare_amount")))
    print("Pandas UDF plan nodes (ArrowEvalPython - columnar batch):")
    show_nodes(pdu_df, keys=("ArrowEvalPython", "BatchEvalPython", "Exchange"))

    _, t_pd = run_and_report("pandas udf", lambda: pdu_df.agg(F.sum("r")).collect())
    print(f"\nbuiltin: {t_b:.2f}s  pandas_udf: {t_pd:.2f}s  python_udf: {t_p:.2f}s")
    print(f"python_udf vs builtin: {t_p/max(t_b,1e-9):.1f}x  pandas_udf vs builtin: {t_pd/max(t_b,1e-9):.1f}x")
except Exception as e:
    print(f"pandas_udf skipped: {e}")

Pandas UDF plan nodes (ArrowEvalPython - columnar batch):
+- ArrowEvalPython [tip_ratio_pd(tip_amount#13, fare_amount#10)#387], [pythonUDF0#419], 200


26/03/25 22:23:01 WARN MemoryStore: Not enough space to cache rdd_70_4 in memory! (computed 19.0 MiB so far)
26/03/25 22:23:01 WARN MemoryStore: Not enough space to cache rdd_70_3 in memory! (computed 19.0 MiB so far)
26/03/25 22:23:01 WARN MemoryStore: Not enough space to cache rdd_70_2 in memory! (computed 10.9 MiB so far)
26/03/25 22:23:01 WARN MemoryStore: Not enough space to cache rdd_70_0 in memory! (computed 18.9 MiB so far)
26/03/25 22:23:01 WARN MemoryStore: Not enough space to cache rdd_70_1 in memory! (computed 29.7 MiB so far)
26/03/25 22:23:01 WARN MemoryStore: Not enough space to cache rdd_70_5 in memory! (computed 29.7 MiB so far)
[Stage 25:======================================>                  (8 + 4) / 12]

pandas udf -> 1.56s jobs=[12], stages=[25, 26], stage_count=2

builtin: 0.27s  pandas_udf: 1.56s  python_udf: 5.13s
python_udf vs builtin: 18.9x  pandas_udf vs builtin: 5.7x


## Troubleshooting checklist

- **Driver OOM**: aggregate first, then `collect()`. Never `collect()` on production-scale data.
- **Executor OOM**: check key distribution and task input sizes before increasing memory.
- **Spill**: increase `spark.sql.shuffle.partitions` or enable AQE coalescing (notebook 05).
- **`BatchEvalPython` in hot path**: replace Python `@udf` with built-in or `@pandas_udf`.

## Key takeaways
- OOM location (driver vs executor) determines the fix - Spark UI first, memory configs last
- Spill is graceful degradation; OOM is a crash - different root causes, different fixes
- UMM gives execution priority over storage - cached data drops under memory pressure
- Python UDFs opt out of Tungsten and WSCG - always try a built-in first